In [3]:
import os
from pathlib import Path

In [4]:
# nltk is used for PDF processing. Here we ensure anything it downloads goes to
# the cache folder, so it doesn't have to download again
nltk_data_path = Path("~/.cache/nltk_data").expanduser()
nltk_data_path.mkdir(parents=True, exist_ok=True)
os.environ["NLTK_DATA"] = str(nltk_data_path)

In [5]:
!pip install chromadb

# Deps for PDF parsing
!pip install "unstructured[pdf]"
!sudo apt-get install -y poppler-utils tesseract-ocr

# I can't even remember why we need this one
!pip install sentence-transformers

# For phi-2
!pip install einops

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.3.1 -> 23.3.2
[notice] To update, run: python3.10 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.3.1 -> 23.3.2
[notice] To update, run: python3.10 -m pip install --upgrade pip
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
poppler-utils is already the newest version (22.02.0-2ubuntu0.3).
0 upgraded, 0 newly installed, 0 to remove and 65 not upgraded.
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.3.1 -> 23.3.2
[notice] To update, run: python3.10 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new r

In [6]:
import os
import json

from typing import Optional, Tuple

import torch
import transformers

from langchain.callbacks.tracers import ConsoleCallbackHandler
from langchain.chains import LLMChain
from langchain.document_loaders import UnstructuredPDFLoader
from langchain.embeddings.huggingface import HuggingFaceEmbeddings
from langchain.llms.huggingface_pipeline import HuggingFacePipeline
from langchain.prompts import PromptTemplate, StringPromptTemplate
from langchain.retrievers import ParentDocumentRetriever
from langchain.schema import AIMessage
from langchain.schema.runnable import RunnablePassthrough
from langchain.storage import InMemoryStore
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import VectorStore
from langchain_community.vectorstores.chroma import Chroma
from langchain_core.language_models import BaseChatModel
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables import Runnable
from langchain_core.runnables import RunnableBranch
from transformers import (
    PreTrainedModel, 
    PreTrainedTokenizerBase,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    AutoModel,
)

In [13]:
%load_ext autoreload
%autoreload 2
from util import HuggingFaceChatModelWithBatchSupport

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
# Configuration

# FAISS
faiss_gpu = False
faiss_embedding_model_name = 'jinaai/jina-embeddings-v2-base-en'
retrieve_topk = 6

# Text splitting settings
chunk_size = 1000
chunk_overlap = 200

# Rag settings
consistency_samples = 5

# Quantization settings
quantization_enabled = False
use_4bit = True
bnb_4bit_compute_dtype = "float16"
bnb_4bit_quant_type = "nf4"
use_nested_quant = True

# Model
# model_name='mistralai/Mistral-7B-Instruct-v0.1'
model_name='teknium/OpenHermes-2.5-Mistral-7B'
# model_name='Intel/neural-chat-7b-v3-1'
# model_name='rishiraj/CatPPT'
# model_name = 'kyujinpy/Sakura-SOLAR-Instruct'

grader_model_name = 'cognitivecomputations/dolphin-2_6-phi-2'

# Data
data_path = Path("./munchkin_rules/")

device = "cuda"

In [9]:
!nvidia-smi

Sat Jan  6 19:56:08 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.129.03             Driver Version: 535.129.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 3090        Off | 00000000:07:00.0  On |                  N/A |
|  0%   55C    P8              44W / 420W |    715MiB / 24576MiB |     29%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [10]:
def load_transformers_model(model_name:str, bnb_config:Optional[BitsAndBytesConfig]=None) -> Tuple[PreTrainedModel, PreTrainedTokenizerBase]:
    # Load model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
        device_map="auto"
    )
    if tokenizer.pad_token is None:
        print("Setting pad token")
        # For some reason, this isn't set in the config. For Mistral, it's just
        # the EOS token (which is the default). However, with OpenHermes, the
        # EOS token is a different token, but the padding token appears to still
        # be </s>:
        #
        # https://huggingface.co/teknium/OpenHermes-2.5-Mistral-7B/blob/main/special_tokens_map.json
        #
        # So if it is not set, we just set it explicitly to </s> here.
        tokenizer.pad_token = '</s>'

    if bnb_config is not None:
        model_kwargs = {"quantization_config": bnb_config}
    else:
        model_kwargs = {"torch_dtype": torch.float16}

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        # OpenHermes has the KV-cache disabled by default (perhaps they setup the
        # defaults for fine-tuning?). Enabling it here because it is way way
        # faster.
        use_cache=True,
        trust_remote_code=True,
        **model_kwargs,
    )

    return (model, tokenizer)


compute_dtype = getattr(torch, bnb_4bit_compute_dtype)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

model, tokenizer = load_transformers_model(model_name, bnb_config)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Setting pad token


Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.96s/it]


In [81]:
chat_model = HuggingFaceChatModelWithBatchSupport.from_model_tokenizer(model=model, tokenizer=tokenizer, batch_size=8)

In [83]:
from langchain.schema import SystemMessage, HumanMessage
chat_model.batch(
    [
        [
            SystemMessage(content="You are a helpful Q&A AI assistant."),
            HumanMessage(content="Give me a brief history of the United States in about 300 words?")
        ],
        [
            SystemMessage(content="You are a helpful Q&A AI assistant."),
            HumanMessage(content="What is the capital of the U.S.?")
        ]
    ],
    do_sample=True,
    temperature=0.7,
    max_new_tokens=1000,
)

[AIMessage(content="The United States of America (USA) is a North American country known for its political influence, economic power, and cultural output. The USA's history is marked by a variety of significant events, including territorial expansion, economic growth, social change, and political development.\n\nThe USA's history can be broadly divided into several periods:\n\n1. Pre-Columbian Era (12,000 BCE - 1492 CE): The land now known as the United States was originally inhabited by Indigenous peoples, who developed sophisticated cultures and societies.\n\n2. European Exploration and Colonization (1492 - 1763): In 1492, Christopher Columbus, an Italian explorer, reached the Americas under the sponsorship of Spain. This marked the beginning of European exploration and colonization. In the early 17th century, English settlers established colonies along the eastern coast, which eventually became 13 British colonies.\n\n3. Revolutionary Era (1763 - 1789): By the mid-18th century, the 

In [80]:
import importlib
import util
importlib.reload(util)
from util import HuggingFaceChatModelWithBatchSupport

In [44]:
conversations = [
    [
        SystemMessage(content="You are a helpful Q&A AI assistant."),
        HumanMessage(content="What is the capital of the U.S.?")
    ]
    for _ in range(100)
]

In [49]:
chat_model.batch(
    conversations,
    do_sample=True,
    temperature=0.7,
)

/usr/local/lib/python3.10/dist-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.


[Conversation id: 89d6464c-31ba-4b31-ae21-b3b87231e6d5
system: You are a helpful Q&A AI assistant.
user: What is the capital of the U.S.?
assistant: The capital of the United States is Washington, D.C. (District of Columbia).
, Conversation id: 32205f6e-c7ae-4b0e-a7f7-da9b98b3f4ca
system: You are a helpful Q&A AI assistant.
user: What is the capital of the U.S.?
assistant: The capital of the United States is Washington, D.C. (District of Columbia).
, Conversation id: bcf79717-fc46-4866-b069-7189028f5b34
system: You are a helpful Q&A AI assistant.
user: What is the capital of the U.S.?
assistant: The capital of the United States is Washington, D.C. (District of Columbia).
, Conversation id: b9829ef0-fe63-496d-b47d-de5cd4aac89b
system: You are a helpful Q&A AI assistant.
user: What is the capital of the U.S.?
assistant: The capital of the United States is Washington, D.C. (District of Columbia).
, Conversation id: b68760a7-4edb-4176-88b3-5763346317eb
system: You are a helpful Q&A AI assi

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.


[Conversation id: 4e17b458-c172-4811-87eb-23f0f9d61f45
system: You are a helpful Q&A AI assistant.
user: What is the capital of the U.S.?
assistant: The capital of the United States is Washington, D.C. (District of Columbia).
, Conversation id: 40a7620a-5c91-4191-90bb-39d61e01db99
system: You are a helpful Q&A AI assistant.
user: What is the capital of the U.S.?
assistant: The capital of the United States is Washington, D.C. (District of Columbia).
, Conversation id: 83f7734f-6a6e-4ed9-be80-b871eca6fb1c
system: You are a helpful Q&A AI assistant.
user: What is the capital of the U.S.?
assistant: The capital of the United States is Washington, D.C. (District of Columbia).
, Conversation id: 95b63b8c-289f-4017-9954-61536df3bdd3
system: You are a helpful Q&A AI assistant.
user: What is the capital of the U.S.?
assistant: The capital of the United States is Washington, D.C. (District of Columbia).
, Conversation id: 467a29b4-bae2-41d7-80fa-094336b19049
system: You are a helpful Q&A AI assi

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.


...

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.


[Conversation id: 59e526cc-04af-4cee-8b35-9426e5f339a9
system: You are a helpful Q&A AI assistant.
user: What is the capital of the U.S.?
assistant: The capital of the United States is Washington, D.C. (District of Columbia).
, Conversation id: 1e721841-2608-4c9e-8216-14dba54a5d4a
system: You are a helpful Q&A AI assistant.
user: What is the capital of the U.S.?
assistant: The capital of the United States is Washington, D.C. (District of Columbia).
, Conversation id: 99fdce70-91e9-4772-895a-38809745a31b
system: You are a helpful Q&A AI assistant.
user: What is the capital of the U.S.?
assistant: The capital of the United States is Washington, D.C. (District of Columbia).
, Conversation id: fa18e9dc-3378-4ef5-b5d8-0c547234ccc1
system: You are a helpful Q&A AI assistant.
user: What is the capital of the U.S.?
assistant: The capital of the United States is Washington, D.C. (District of Columbia).
, Conversation id: b05096d6-c790-4fcc-9fcd-abe133d9a80e
system: You are a helpful Q&A AI assi

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.


[Conversation id: b46d11df-a884-499f-b9e6-46032c7a41ea
system: You are a helpful Q&A AI assistant.
user: What is the capital of the U.S.?
assistant: The capital of the United States is Washington, D.C. (District of Columbia).
, Conversation id: 1118f4ab-750c-4d65-bd23-eae37ffac108
system: You are a helpful Q&A AI assistant.
user: What is the capital of the U.S.?
assistant: The capital of the United States is Washington, D.C. (District of Columbia).
, Conversation id: 0c1f43c8-db0f-4445-801b-1f6d2b209f19
system: You are a helpful Q&A AI assistant.
user: What is the capital of the U.S.?
assistant: The capital of the United States is Washington, D.C. (District of Columbia).
, Conversation id: 2bfa7a1c-0fef-4d72-97b7-2f52c94089d6
system: You are a helpful Q&A AI assistant.
user: What is the capital of the U.S.?
assistant: The capital of the United States is Washington, D.C. (District of Columbia).
, Conversation id: ede89354-ba33-46d4-b0c7-134813e3e4d6
system: You are a helpful Q&A AI assi

[AIMessage(content='The capital of the United States is Washington, D.C. (District of Columbia).'),
 AIMessage(content='The capital of the United States is Washington, D.C. (District of Columbia).'),
 AIMessage(content='The capital of the United States is Washington, D.C. (District of Columbia).'),
 AIMessage(content='The capital of the United States is Washington, D.C. (District of Columbia).'),
 AIMessage(content='The capital of the United States is Washington, D.C. (District of Columbia).'),
 AIMessage(content='The capital of the United States is Washington, D.C. (District of Columbia).'),
 AIMessage(content='The capital of the United States is Washington, D.C. (District of Columbia).'),
 AIMessage(content='The capital of the United States is Washington, D.C. (District of Columbia).'),
 AIMessage(content='The capital of the United States is Washington, D.C. (District of Columbia).'),
 AIMessage(content='The capital of the United States is Washington, D.C. (District of Columbia).'),


In [46]:
[
    chat_model.invoke(conversation)
    for conversation in conversations
]

/usr/local/lib/python3.10/dist-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.
/usr/local/lib/python3.10/dist-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.


Conversation id: 5599e8c8-8056-4a9e-aaf1-01a16d471d0c
system: You are a helpful Q&A AI assistant.
user: What is the capital of the U.S.?
assistant: The capital of the United States is Washington, D.C. (District of Columbia).

Conversation id: 5599e8c8-8056-4a9e-aaf1-01a16d471d0c
system: You are a helpful Q&A AI assistant.
user: What is the capital of the U.S.?
assistant: The capital of the United States is Washington, D.C. (District of Columbia).



/usr/local/lib/python3.10/dist-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.


Conversation id: b0147e83-6a9f-4bff-b9f1-58a7f16ef898
system: You are a helpful Q&A AI assistant.
user: What is the capital of the U.S.?
assistant: The capital of the United States is Washington, D.C. (District of Columbia).

Conversation id: b0147e83-6a9f-4bff-b9f1-58a7f16ef898
system: You are a helpful Q&A AI assistant.
user: What is the capital of the U.S.?
assistant: The capital of the United States is Washington, D.C. (District of Columbia).



/usr/local/lib/python3.10/dist-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.


...

/usr/local/lib/python3.10/dist-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.


Conversation id: b6ccc568-baad-4fa3-ad0a-91a1ec79439b
system: You are a helpful Q&A AI assistant.
user: What is the capital of the U.S.?
assistant: The capital of the United States is Washington, D.C. (District of Columbia).

Conversation id: b6ccc568-baad-4fa3-ad0a-91a1ec79439b
system: You are a helpful Q&A AI assistant.
user: What is the capital of the U.S.?
assistant: The capital of the United States is Washington, D.C. (District of Columbia).



/usr/local/lib/python3.10/dist-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:32000 for open-end generation.


Conversation id: 3dd25f3d-b3d9-490a-9a3d-188681884d7f
system: You are a helpful Q&A AI assistant.
user: What is the capital of the U.S.?
assistant: The capital of the United States is Washington, D.C. (District of Columbia).

Conversation id: 3dd25f3d-b3d9-490a-9a3d-188681884d7f
system: You are a helpful Q&A AI assistant.
user: What is the capital of the U.S.?
assistant: The capital of the United States is Washington, D.C. (District of Columbia).

Conversation id: b065cfb2-a4a3-460c-a0c6-9b13157ef930
system: You are a helpful Q&A AI assistant.
user: What is the capital of the U.S.?
assistant: The capital of the United States is Washington, D.C. (District of Columbia).

Conversation id: b065cfb2-a4a3-460c-a0c6-9b13157ef930
system: You are a helpful Q&A AI assistant.
user: What is the capital of the U.S.?
assistant: The capital of the United States is Washington, D.C. (District of Columbia).



[AIMessage(content='The capital of the United States is Washington, D.C. (District of Columbia).'),
 AIMessage(content='The capital of the United States is Washington, D.C. (District of Columbia).'),
 AIMessage(content='The capital of the United States is Washington, D.C. (District of Columbia).'),
 AIMessage(content='The capital of the United States is Washington, D.C. (District of Columbia).'),
 AIMessage(content='The capital of the United States is Washington, D.C. (District of Columbia).'),
 AIMessage(content='The capital of the United States is Washington, D.C. (District of Columbia).'),
 AIMessage(content='The capital of the United States is Washington, D.C. (District of Columbia).'),
 AIMessage(content='The capital of the United States is Washington, D.C. (District of Columbia).'),
 AIMessage(content='The capital of the United States is Washington, D.C. (District of Columbia).'),
 AIMessage(content='The capital of the United States is Washington, D.C. (District of Columbia).'),
